# 🎼 Audio → MIDI (BasicPitch) 
This notebook converts audio files (e.g., mp3/wav) to MIDI using **BasicPitch** and **PrettyMIDI**, then visualizes a simple piano-roll. 

**Requirements:**
- Python 3.9+ recommended
- `pip install basic-pitch[tf] pretty_midi librosa soundfile matplotlib`

**Inputs:** Provide a path to an audio file in `INPUT_AUDIO`.
**Outputs:** `.mid` and `.json` note output files in the `OUTPUT_DIR`.

In [ ]:
# Install (uncomment if running locally)
# !pip install --upgrade pip
# !pip install basic-pitch[tf] pretty_midi librosa soundfile matplotlib


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
import pretty_midi as pm
import librosa
import librosa.display

INPUT_AUDIO = 'path/to/your/audio.wav'  # <-- change me
OUTPUT_DIR = Path('outputs/midi')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Using model:', ICASSP_2022_MODEL_PATH)
pred, midi, note_events = predict(
    [INPUT_AUDIO],
    model_or_model_path=ICASSP_2022_MODEL_PATH,
    onset_threshold=0.5,
    frame_threshold=0.3,
    use_onset_prob=True,
)

# Save MIDI & JSON
midi_path = OUTPUT_DIR / (Path(INPUT_AUDIO).stem + '.mid')
json_path = OUTPUT_DIR / (Path(INPUT_AUDIO).stem + '.json')

midi[0].write(str(midi_path))
with open(json_path, 'w') as f:
    json.dump(note_events[0], f, indent=2)

print('Saved:', midi_path)
print('Saved:', json_path)

## 🎹 Visualize Piano Roll

In [ ]:
m = pm.PrettyMIDI(str(midi_path))
piano_roll = m.get_piano_roll(fs=50)  # frames per second
fig = plt.figure(figsize=(12,4))
plt.imshow(piano_roll, origin='lower', aspect='auto')
plt.xlabel('Time (frames)')
plt.ylabel('MIDI Pitch')
plt.title('Piano Roll (velocity)')
plt.show()

# Optional: overlay waveform
y, sr = librosa.load(INPUT_AUDIO, sr=None)
plt.figure(figsize=(12,2))
librosa.display.waveshow(y, sr=sr)
plt.title('Waveform')
plt.tight_layout()
plt.show()